# 07 — Fine-tuning IndoBERT (Jalur B) — Colab / Kaggle

Deep learning pada kolom **`text_bert`** (cleaning minimal, morfologi terjaga).
Notebook ini **self-contained**: tidak butuh repo, cukup unggah 3 file parquet hasil
notebook `05_build_dataset.ipynb`.

**Sebelum mulai:** Runtime → Change runtime type → **GPU** (T4 cukup).

Langkah:
1. Install dependency.
2. Unggah `train.parquet`, `val.parquet`, `test.parquet`.
3. Tokenisasi → fine-tune `indobenchmark/indobert-base-p1` (3 kelas).
4. Evaluasi di test (macro-F1 + confusion matrix), simpan model & metrik.

> Dataset seimbang (1.000/kelas) → tidak perlu weighted loss. Jika nanti memakai
> dataset timpang, lihat sel opsional di akhir.

In [ ]:
!pip -q install transformers datasets evaluate accelerate scikit-learn

In [ ]:
import torch
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("PERINGATAN: tidak ada GPU. Aktifkan GPU runtime, fine-tuning di CPU sangat lambat.")

In [ ]:
# Unggah split. Taruh file di /content (file browser) ATAU jalankan sel ini untuk upload.
import os, pandas as pd
SPLIT_DIR = "/content"
need = ["train.parquet", "val.parquet", "test.parquet"]
missing = [f for f in need if not os.path.exists(os.path.join(SPLIT_DIR, f))]
if missing:
    from google.colab import files
    print("Unggah file berikut:", missing)
    files.upload()

df_train = pd.read_parquet(f"{SPLIT_DIR}/train.parquet")
df_val   = pd.read_parquet(f"{SPLIT_DIR}/val.parquet")
df_test  = pd.read_parquet(f"{SPLIT_DIR}/test.parquet")
print({k: len(v) for k, v in [("train", df_train), ("val", df_val), ("test", df_test)]})
df_train[["text_bert", "label", "label_id"]].head()

In [ ]:
# Konstanta label (HARUS sama urutannya dengan src/modeling/labels.py)
LABELS = ["Negatif", "Netral", "Positif"]
id2label = {i: l for i, l in enumerate(LABELS)}
label2id = {l: i for i, l in enumerate(LABELS)}
MODEL_NAME = "indobenchmark/indobert-base-p1"
MAX_LEN = 128
SEED = 42

import numpy as np, random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(df):
    d = Dataset.from_pandas(
        df[["text_bert", "label_id"]].rename(columns={"label_id": "labels"}),
        preserve_index=False,
    )
    return d

def tok(batch):
    return tokenizer(batch["text_bert"], truncation=True, max_length=MAX_LEN)

ds_train = to_ds(df_train).map(tok, batched=True)
ds_val   = to_ds(df_val).map(tok, batched=True)
ds_test  = to_ds(df_test).map(tok, batched=True)

In [ ]:
from transformers import AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
    }

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback

collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Nama argumen evaluasi berubah antar versi transformers (evaluation_strategy ->
# eval_strategy). Bangun kwargs lalu coba keduanya agar portabel.
base_kwargs = dict(
    output_dir="indobert-out",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
try:
    args = TrainingArguments(eval_strategy="epoch", save_strategy="epoch", **base_kwargs)
except TypeError:
    args = TrainingArguments(evaluation_strategy="epoch", save_strategy="epoch", **base_kwargs)

trainer = Trainer(
    model=model, args=args,
    train_dataset=ds_train, eval_dataset=ds_val,
    tokenizer=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)
trainer.train()

In [ ]:
# Evaluasi final di TEST — skema metrik dibuat SAMA dengan src/modeling/evaluate.py
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(ds_test)
y_pred = np.argmax(pred_out.predictions, axis=-1)
y_true = np.array(ds_test["labels"])

rep = classification_report(y_true, y_pred, labels=list(range(len(LABELS))),
                            target_names=LABELS, output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(LABELS)))).tolist()

metrics = {
    "model": "IndoBERT",
    "accuracy": round(accuracy_score(y_true, y_pred), 4),
    "macro_f1": round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4),
    "weighted_f1": round(f1_score(y_true, y_pred, average="weighted", zero_division=0), 4),
    "per_class": {l: {"precision": round(rep[l]["precision"], 4),
                      "recall": round(rep[l]["recall"], 4),
                      "f1": round(rep[l]["f1-score"], 4),
                      "support": int(rep[l]["support"])} for l in LABELS},
    "confusion_matrix": cm,
    "labels": LABELS,
}
print(f"Accuracy : {metrics['accuracy']:.4f}")
print(f"Macro-F1 : {metrics['macro_f1']:.4f}  <-- metrik utama")
print(f"Weighted : {metrics['weighted_f1']:.4f}")
for l in LABELS:
    pc = metrics["per_class"][l]
    print(f"  {l:<8} P={pc['precision']:.3f} R={pc['recall']:.3f} F1={pc['f1']:.3f} (n={pc['support']})")
print("Confusion matrix (baris=aktual):"); [print(" ", LABELS[i], r) for i, r in enumerate(cm)]

In [ ]:
# Plot confusion matrix
import matplotlib.pyplot as plt
arr = np.array(cm)
fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(arr, cmap="Blues")
ax.set_xticks(range(len(LABELS)), LABELS); ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("Prediksi"); ax.set_ylabel("Aktual"); ax.set_title("IndoBERT — Test")
th = arr.max() / 2 if arr.max() else 0
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, str(arr[i, j]), ha="center", va="center",
                color="white" if arr[i, j] > th else "black")
fig.colorbar(im, ax=ax, fraction=0.046); fig.tight_layout()
fig.savefig("indobert_test_confusion.png", dpi=120); plt.show()

In [ ]:
# Simpan model, tokenizer, metrik; lalu unduh
import json
trainer.save_model("indobert-jokowi")
tokenizer.save_pretrained("indobert-jokowi")
with open("indobert_test_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

# Zip model untuk diunduh (opsional — model ~450MB)
!zip -qr indobert-jokowi.zip indobert-jokowi
from google.colab import files
files.download("indobert_test_metrics.json")
files.download("indobert_test_confusion.png")
# files.download("indobert-jokowi.zip")  # uncomment untuk unduh bobot model

## Bandingkan dengan SVM

Letakkan `indobert_test_metrics.json` dan `svm_test_metrics.json` (dari notebook 06)
di `outputs/reports/`, lalu jalankan di lokal:

```python
import json
from src.modeling.evaluate import compare_models
svm = json.load(open("outputs/reports/svm_test_metrics.json"))
bert = json.load(open("outputs/reports/indobert_test_metrics.json"))
compare_models({"SVM+TF-IDF": svm, "IndoBERT": bert})
```

---
### (Opsional) Weighted loss untuk dataset timpang
Jika kelak memakai dataset tidak seimbang, ganti `Trainer` dengan subclass yang
menimpa `compute_loss` memakai `nn.CrossEntropyLoss(weight=class_weights)` (bobot =
inverse frekuensi kelas pada train).